# Notebook 4 - Penyusunan Dataset Tanpa Fitur DDS

**Tahap diagram:** Ekstraksi data dan DDS -> dataset tanpa fitur trik DDS -> split data.

Notebook ini menyiapkan dan memeriksa dataset dengan **164 fitur kanonik** tanpa fitur jumlah trik DDS. Label `matches_par_contract` tetap dihitung menggunakan hasil DDS. Notebook ini tidak melatih atau mengevaluasi model; seluruh pemodelan dilakukan di Notebook 5.


---
## 0. Setup


In [14]:
from pathlib import Path
import json
import sys
import subprocess

import pandas as pd

_candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next(p for p in _candidates if (p / "configs/config.yaml").exists() and (p / "src").is_dir())
sys.path.insert(0, str(ROOT))

DATA_NO_DDS = ROOT / "data" / "processed_no_dds"
DATA_WITH_DDS = ROOT / "data" / "processed"
FEATURES_NO_DDS = DATA_NO_DDS / "feature_columns.json"
FEATURES_WITH_DDS = DATA_WITH_DDS / "feature_columns.json"

if not FEATURES_NO_DDS.exists():
    print("Membangun dataset tanpa fitur DDS...")
    subprocess.run(
        [sys.executable, str(ROOT / "scripts" / "build_dataset_no_dds.py")],
        cwd=ROOT, check=True,
    )

for required in [FEATURES_NO_DDS, FEATURES_WITH_DDS]:
    if not required.exists():
        raise FileNotFoundError(f"File fitur tidak ditemukan: {required}")

print(f"Repo root: {ROOT}")
print(f"Dataset tanpa DDS: {DATA_NO_DDS}")


Repo root: d:\Bridge-Prediction
Dataset tanpa DDS: d:\Bridge-Prediction\data\processed_no_dds


---
## 1. Muat Dataset Tanpa Fitur DDS

Muat partisi train, validation, dan test yang telah dibangun dengan identitas deal kanonik. Dataset pembanding dengan fitur DDS hanya dibaca untuk memastikan label dan pembagian datanya tetap sama.


In [15]:
KEYS = ["_source_file", "_room", "_board_number", "_deal_key"]
FEATURES = json.loads(FEATURES_NO_DDS.read_text(encoding="utf-8"))
FEATURES_WITH = json.loads(FEATURES_WITH_DDS.read_text(encoding="utf-8"))
DDS_PREFIXES = ("ns_dd_", "ew_dd_", "dd_par_")
DDS_TRICK_FEATURES = [name for name in FEATURES_WITH if name.startswith(("ns_dd_", "ew_dd_"))]

no_dds = {}
with_dds = {}
for split_name in ("train", "val", "test"):
    no_dds[split_name] = pd.read_csv(
        DATA_NO_DDS / f"{split_name}.csv", keep_default_na=False,
        dtype={key: str for key in KEYS},
    )
    with_dds[split_name] = pd.read_csv(
        DATA_WITH_DDS / f"{split_name}.csv", keep_default_na=False,
        dtype={key: str for key in KEYS},
    )

    a = no_dds[split_name]
    b = with_dds[split_name]
    assert len(a) == len(b), f"Jumlah baris berbeda pada split {split_name}"
    assert a[KEYS + ["label"]].equals(b[KEYS + ["label"]]), (
        f"Identitas board, label, atau urutan split berbeda pada {split_name}"
    )

assert len(FEATURES) == 164, f"Diharapkan 164 fitur; ditemukan {len(FEATURES)}"
assert not any(name.startswith(DDS_PREFIXES) for name in FEATURES)
assert len(DDS_TRICK_FEATURES) == 10, f"Diharapkan 10 fitur trik DDS; ditemukan {len(DDS_TRICK_FEATURES)}"
assert set(FEATURES).issubset(no_dds["train"].columns)
assert "matches_par_contract" in no_dds["train"].columns

print("Jumlah fitur kanonik:", len(FEATURES))
print("Fitur trik DDS yang dikeluarkan:", len(DDS_TRICK_FEATURES))
print("Label dan identitas/split sama dengan dataset berfitur DDS.")


Jumlah fitur kanonik: 164
Fitur trik DDS yang dikeluarkan: 10
Label dan identitas/split sama dengan dataset berfitur DDS.


---
## 2. Ringkasan Partisi dan Target

Tabel ini memastikan ukuran partisi dan proporsi kelas positif pada dataset yang akan digunakan Notebook 5.


In [16]:
rows = []
for split_name, frame in no_dds.items():
    rows.append({
        "split": split_name,
        "jumlah_board": len(frame),
        "fitur_model": len(FEATURES),
        "jumlah_cocok_par": int(frame["matches_par_contract"].sum()),
        "proporsi_cocok_par": float(frame["matches_par_contract"].mean()),
        "jumlah_deal_unik": frame["_deal_key"].nunique(),
    })

dataset_summary = pd.DataFrame(rows).set_index("split")
dataset_summary.style.format({"proporsi_cocok_par": "{:.2%}"})


,jumlah_board,fitur_model,jumlah_cocok_par,proporsi_cocok_par,jumlah_deal_unik
split,,,,,
train,35161,164,8635,24.56%,15990
val,7534,164,1851,24.57%,3402
test,7532,164,1849,24.55%,3400


---
## 3. Pemeriksaan Dataset untuk Pemodelan

Setiap fitur input berasal dari tangan, auction, atau kondisi board. Kolom DDS dipakai saat membentuk label, tetapi tidak dimasukkan sebagai fitur model. Notebook 5 memuat split ini untuk melatih Random Forest, XGBoost, dan LightGBM.


In [17]:
feature_groups = {
    "fitur kanonik": len(FEATURES),
    "fitur jumlah trik DDS": len(DDS_TRICK_FEATURES),
    "jumlah board": sum(len(frame) for frame in no_dds.values()),
    "positif keseluruhan": sum(int(frame["matches_par_contract"].sum()) for frame in no_dds.values()),
}
print(pd.Series(feature_groups, name="jumlah"))
print("\nDataset untuk modeling:", DATA_NO_DDS)


fitur kanonik              164
fitur jumlah trik DDS       10
jumlah board             50227
positif keseluruhan      12335
Name: jumlah, dtype: int64

Dataset untuk modeling: d:\Bridge-Prediction\data\processed_no_dds
